# Лекция 11. Прямые методы решения плотных СЛУ и декомпозиция матриц


In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(linewidth=120, precision=4, suppress=True)


## 1. 🎯 Постановка задачи и проклятие формул Крамера

Решаем **систему $n$ линейных алгебраических уравнений** (СЛАУ)

$$a_{i1}x_1 + a_{i2}x_2 + \dots + a_{in}x_n = b_i, \qquad i = 1,\dots,n
\quad\Longleftrightarrow\quad Ax = b.$$

Все методы её решения делятся на два класса:

- **прямые** — приводят к решению за конечное число арифметических
  операций (следствие: в идеальной арифметике ответ *точный*);
- **итерационные** — строят сходящуюся последовательность приближений.

По формулам Крамера $x_i = \det A_i / \det A$ решение выражается явно,
но вычислительно это провал: нужно $n+1$ определителей, а вычисление
каждого определителя разложениями по строке требует порядка $n!$
умножений. Итого $\sim (n+1)\cdot n!$: при $n = 20$ — порядка
$5\cdot 10^{19}$ операций, при $n = 30$ — уже $\sim 10^{33}$, при
$n = 100$ — $\sim 10^{160}$. Прямой ход метода Гаусса обрабатывает ту же
матрицу за $\approx n^3/3$ умножений и делений — при $n = 20$ это всего
$\sim 2700$ операций.


## 2. 📏 Нормы векторов и матриц: чем измеряем погрешности

Норма — способ измерять «величину» вектора: функция $\|x\|$ со свойствами
(1) $\|x\| \ge 0$ и $\|x\| = 0 \Leftrightarrow x = 0$;
(2) $\|\alpha x\| = |\alpha|\,\|x\|$;
(3) $\|x+y\| \le \|x\| + \|y\|$.

Базовые векторные нормы:

$$\|x\|_1 = \sum_{i=1}^n |x_i|, \qquad
\|x\|_2 = \Big(\sum_{i=1}^n x_i^2\Big)^{1/2}, \qquad
\|x\|_\infty = \max_{1\le i\le n}|x_i|,$$

и они согласованы между собой (важнейшие две цепочки):

$\ \|x\|_\infty \le \|x\|_2 \le \sqrt{n}\,\|x\|_\infty,
\qquad \|x\|_2 \le \|x\|_1 \le \sqrt{n}\,\|x\|_2.$

**Матричные нормы** — операторные (согласованные): $\|A\| = \max_{x\ne 0}
\|Ax\|/\|x\|$, так что $\|Ax\| \le \|A\|\,\|x\|$ выполняется автоматически:

$$\|A\|_1 = \max_{1\le j\le n}\sum_{i=1}^n |a_{ij}|
   \ \text{ (максимальная сумма по столбцам)},$$
$$\|A\|_\infty = \max_{1\le i\le n}\sum_{j=1}^n |a_{ij}|
   \ \text{ (максимальная сумма по строкам)},$$
$$\|A\|_2 = \sqrt{\lambda_{\max}(A^\top A)} = \sigma_{\max}(A)
   \ \text{ (максимальное сингулярное число)}.$$

Через нормы записываются все оценки точности этой лекции, включая
множитель ошибок $\mathrm{cond}(A) = \|A\|\|A^{-1}\|$ (§3).


In [2]:
# вычислительно: нормы на сквозной матрице примера 2.1
A1 = np.array([[2., -1., 1.], [4., 3., 1.], [6., -13., 6.]])
A1inv = np.linalg.inv(A1)
x = np.array([1., 2., 3.])
Ax = A1 @ x

print('нормы вектора x = (1, 2, 3):')
print('  |x|1 =', np.linalg.norm(x, 1),
      ' |x|2 =', round(np.linalg.norm(x, 2), 6),
      ' |x|inf =', np.linalg.norm(x, np.inf))

print('нормы матрицы A (пример 2.1):')
print('  |A|1 =', np.linalg.norm(A1, 1),
      ' (макс. сумма по столбцам; суммы: 12, 17, 8)')
print('  |A|2 =', round(np.linalg.norm(A1, 2), 6),
      ' (= sigma_max =',
      round(np.linalg.svd(A1, compute_uv=False)[0], 6), ')')
print('  |A|inf =', np.linalg.norm(A1, np.inf),
      ' (макс. сумма по строкам; суммы: 4, 8, 25)')

print('согласованность: Ax =', Ax.tolist(),
      ' |Ax|inf =', np.linalg.norm(Ax, np.inf),
      '<= |A|inf * |x|inf =',
      np.linalg.norm(A1, np.inf) * np.linalg.norm(x, np.inf))

print('нормы обратной матрицы (понадобятся в cond):')
print('  |A^-1|1 =', np.linalg.norm(A1inv, 1),
      ' |A^-1|2 =', round(np.linalg.norm(A1inv, 2), 6),
      ' |A^-1|inf =', np.linalg.norm(A1inv, np.inf))


нормы вектора x = (1, 2, 3):
  |x|1 = 6.0  |x|2 = 3.741657  |x|inf = 3.0
нормы матрицы A (пример 2.1):
  |A|1 = 17.0  (макс. сумма по столбцам; суммы: 12, 17, 8)
  |A|2 = 15.662821  (= sigma_max = 15.662821 )
  |A|inf = 25.0  (макс. сумма по строкам; суммы: 4, 8, 25)
согласованность: Ax = [3.0, 13.0, -2.0]  |Ax|inf = 13.0 <= |A|inf * |x|inf = 75.0
нормы обратной матрицы (понадобятся в cond):
  |A^-1|1 = 11.900000000000006  |A^-1|2 = 8.237704  |A^-1|inf = 10.000000000000004


## 3. 🌡️ Обусловленность СЛАУ: множитель ошибок $\mathrm{cond}(A)$

Пусть данными при решении $Ax = b$ стал «испорченный» вектор
$b + \Delta b$ (например, из-за машинного округления). Решение тогда
$x + \Delta x$, где $\Delta x = A^{-1}\Delta b$ — матрица задачи не
меняется! Для относительных погрешностей отсюда стандартная оценка
(теорема об обусловленности):

$$\frac{\|\Delta x\|}{\|x\|} \;\le\;
\underbrace{\|A\|\;\|A^{-1}\|}_{=\ \mathrm{cond}(A)}\;
\frac{\|\Delta b\|}{\|b\|},
\qquad \mathrm{cond}(A) = \|A\|\|A^{-1}\| \ge 1.$$

Свойства и смысл:

- $\mathrm{cond}(A) \ge 1$ всегда; «хорошие» системы: cond порядка
  единиц–десятков; тогда малые относительные ошибки данных дают малые
  относительные ошибки ответа.
- $\mathrm{cond}_2(A) = \sigma_1/\sigma_n$ — отношение крайних
  сингулярных чисел (см. §6); $\mathrm{cond}_1, \mathrm{cond}_\infty$
  — с соответствующими нормами.
- **Плохо обусловленная** (ill-conditioned) система: cond огромен —
  малые ошибки в данных усиливаются в тысячи–миллионы раз. Если
  $\mathrm{cond}(A) \gtrsim 1/\varepsilon_{\text{mach}}$ (при double
  $\varepsilon \approx 10^{-16}$), то даже точный метод не гарантирует
  ни одной верной цифры ответа.

**Невязка — не ошибка.** Невязка $r = b - Ax^\wedge$ показывает, какому
*возмущённому* $b$ найденный $x^\wedge$ отвечает точно; при большом
cond малая невязка не мешает巨大 ошибке $\|x^\wedge - x\|$.

**Итерационное уточнение (§ 2.8.1 учебника).** Дешёвый способ подстегнуть
точность прямых методов: решив $Ax^{(0)} = b$ (прямой ход уже сделан и
хранится в LU), считаем невязку $r^{(0)} = b - Ax^{(0)}$, находим
поправку $p$ из $Ap = r^{(0)}$ (только обратные подстановки) и положим
$x^{(1)} = x^{(0)} + p$; повторяем 1–3 раза. Если невязки не
сходятся — система плохо обусловлена и без дополнительных данных о
задаче её нельзя решить требуемым образом.


## 4. 🧮 Метод Гаусса и LU-разложение

### 4.1 Метод Гаусса (схема единственного деления)

Прямой ход: последовательно для $k = 1,\dots,n-1$ исключаем $x_k$ из
строк $i > k$:

$$l_{ik} := \frac{a_{ik}^{(k-1)}}{a_{kk}^{(k-1)}}, \qquad
a_{ij}^{(k)} := a_{ij}^{(k-1)} - l_{ik}\,a_{kj}^{(k-1)}, \qquad i > k,\ j \ge k.$$

Обратный ход — обратные подстановки при $i = n,\dots,1$:

$$x_i = \frac{b_i^{(k-1)} - \sum_{j=i+1}^{n}a_{ij}^{(k-1)}x_j}{a_{ii}^{(k-1)}}.$$

Стоимость: $\approx n^3/3 + n^2$ умножений и делений. Числа, на которые
делим, — **ведущие (главные) элементы**; определитель равен их
произведению:

$$\det A = \prod_{k=1}^{n} a_{kk}^{(k)},$$

поскольку вычитание из строк кратных других строк не меняет
определитель, а треугольный вид даёт произведение диагонали.


### 4.2 LU-разложение (компактная схема Гаусса)

**Теорема 2.1.** Если все главные миноры
$\Delta_k = \det(a_{ij})_{i,j \le k}$, $k = 1,\dots,n$, матрицы $A$
отличны от нуля, то существуют нижняя $L$ и верхняя $U$ треугольные
матрицы: $A = LU$; при фиксированной диагонали одной из них (например
$l_{ii} = 1$) разложение **единственно**.
Достаточное условие — **диагональное преобладание**
$|a_{ii}| > \sum_{j \ne i}|a_{ij}|$: тогда все главные миноры ненулевые
автоматически.

Компактная схема вычисляет $L, U$ **напрямую** (без промежуточных
матриц), поочерёдно строка $U$ — столбец $L$ (формулы (2.9)–(2.10)):

$$U_{1j} = a_{1j};\qquad
U_{ij} = a_{ij} - \sum_{k=1}^{i-1}l_{ik}U_{kj}\ \ (i \le j);$$
$$L_{ij} = \frac{a_{ij} - \sum_{k=1}^{j-1}l_{ik}U_{kj}}{U_{jj}}\ \ (i > j),
\qquad l_{ii} = 1.$$

Решение системы через LU  — две треугольные
системы $Ly = b$, $Ux = y$:

$$y_i = b_i - \sum_{k=1}^{i-1}l_{ik}y_k, \qquad
x_i = \frac{y_i - \sum_{k=i+1}^{n}u_{ik}x_k}{u_{ii}}\ (i = n,\dots,1).$$

При $l_{ii} = 1$: $\det A = \prod_i u_{ii}$ (ведущие элементы = диагональ
$U$). Компактная схема — «та же схема Гаусса по-другому организованная».

### 4.3 Обращение через LU: формулы 

Из $AX = E$ по столбцам $Ax_j = e_j$; при решении «столбцами справа
налево»-elements обратной матрицы вычисляются по формулам Холецкого:

$$x_{nn} = \frac{1}{u_{nn}},\qquad
x_{ii} = \frac{1-\sum_{k=i+1}^{n}u_{ik}x_{ki}}{u_{ii}}\ \text{(диагональ)},$$
$$x_{ij} = -\frac{1}{u_{ii}}\sum_{k=i+1}^{n}u_{ik}x_{kj}\ (i<j), \qquad
x_{ij} = -\sum_{k=j+1}^{n}l_{kj}x_{ik}\ (i>j).$$

## 5. ⬇️ Метод квадратных корней (схема Холецкого) для симметричных систем

Если $A = A^\top$, вычислять полный LU расточительно — содержит двойную
работу. Для решения ищем разложение в форме $A = U^\top U$ (или в
нижнетреугольной записи $A = LL^\top$, так работает np.linalg.cholesky).
Для **положительно определённой** ($x^\top A x > 0$ при $x \neq 0$;
эквивалентно: все главные миноры $> 0$) матрицы существует вещественная
верхнетреугольная $U$ с положительной диагональю; её элементы —
формулы:

$$u_{11} = \sqrt{a_{11}}, \qquad
u_{jj} = \sqrt{a_{jj} - \sum_{k=1}^{j-1}u_{kj}^2},$$
$$u_{ij} = \frac{a_{ij} - \sum_{k=1}^{i-1}u_{ki}u_{kj}}{u_{ii}},
\qquad i < j.$$

Правая часть не участвует в разложении; решение = две треугольные
системы, формулы (2.19)–(2.20):

$$U^\top y = b: \quad
y_i = \frac{b_i - \sum_{k=1}^{i-1}u_{ki}y_k}{u_{ii}}, \qquad
U x = y: \quad
x_i = \frac{y_i - \sum_{k=i+1}^{n}u_{ik}x_k}{u_{ii}}\ (i = n,\dots,1).$$

Простые свойства:

- $\det A = \prod_j u_{jj}^2$ (вдвое дешевле, чем произведение диагоналей
  полного LU: ведь каждая $u_{jj}$ «стоит» за два множителя);
- сложность: $\approx n^3/6$ умножений/делений (вдвое дешевле метода
  Гаусса — считается только верхний треугольник), плюс $n$ извлечений
  корня;
- если матрица **не** положительно определена, подкоренное выражение на
  каком-то шаге станет отрицательным (появятся мнимые $u_{jj}$) —
  простейший тест на SPD; при реализации стоит проверять немалость
  $|u_{jj}|$, чтобы не накапливать погрешности.


## 6. 🔄 Метод вращений 

**Почему вращения.** Гауссово исключение использует *неортогональные*
комбинации строк — элементы промежуточных матриц могут расти.
Классический пример [74] из учебника:

$$A = \begin{pmatrix}1&0&0&1\\-1&1&0&1\\-1&-1&1&1\\-1&-1&-1&1\end{pmatrix}:
\ \text{последний диагональный элемент при прямом ходе Гаусса проходит}
\ 2 \to 4 \to 8,$$

элементы матрицы растут до $2^{n-1}$ (при $n = 4$ — до $8$).

**Идея метода вращений (Гивенса).** Обнулять поддиагональный элемент не
вычитанием кратной строки, а ортогональным поворотом: пару уравнений
заменяем на $c_1\cdot\text{строка}_1 + s_1\cdot\text{строка}_2$ и
$-s_1\cdot\text{строка}_1 + c_1\cdot\text{строка}_2$, где (формулы (2.25))

$$c_1 = \frac{a_{11}}{\sqrt{a_{11}^2 + a_{21}^2}}, \qquad
s_1 = \frac{a_{21}}{\sqrt{a_{11}^2 + a_{21}^2}}, \qquad
c_1^2 + s_1^2 = 1$$

— косинус и синус угла поворота (отсюда название). Ключевое свойство:
для любой пары элементов одного столбца сумма квадратов сохраняется:
$(c\,a + s\,b)^2 + (-s\,a + c\,b)^2 = a^2 + b^2$, поэтому **нормы всех
столбцов не меняются и роста элементов нет вообще** — всякий
промежуточный шаг рассматривается как вращение на угол $\alpha_1$
расширенной матрицы системы. Цена: в $\approx 4n^3/3$ умножений
(вчетверо дороже метода Гаусса), кроме того коэффициенты первого
уравнения пересчитываются $n - 1$ раз; зато устойчивость максимальная.


## 7. ➡️ Трёхдиагональные системы: метод прогонки

Класс систем с **ленточными** матрицами - наиболее важный частный случай


$$b_ix_{i-1} + c_ix_i + d_ix_{i+1} = r_i, \qquad i = 1,\dots,n,
\qquad b_1 = d_n = 0. \tag{2.21}$$


Идея **метода прогонки** (алгоритм Томаса): ищем решение в виде прямой
рекуррентности

$$x_i = \delta_i x_{i+1} + \Lambda_i\ \ (i = 1,\dots,n-1), \qquad
x_n = \Lambda_n, \tag{2.22}$$

где **прогоночные коэффициенты** вычисляются прямым проходом:

$$A_i = c_i + b_i\delta_{i-1},\quad
\delta_i = -\frac{d_i}{A_i},\quad
\Lambda_i = \frac{r_i - b_i\Lambda_{i-1}}{A_i}
\ \ (i = 1,\dots,n), \tag{2.23}$$

с инициализацией $A_1 = c_1$, $\delta_1 = -d_1/c_1$,
$\Lambda_1 = r_1/c_1$, а затем $x_n = \Lambda_n$ и обратным проходом
$x_i = \delta_i x_{i+1} + \Lambda_i$ по $i = n-1,\dots,1$.
Стоимость: $\sim 5n$ умножений/делений против $n^3/3$ у
Гаусса.

Системе соответствует трёхдиагональная матрица,
LU-разложение:

$$A = LU,\quad L = \begin{pmatrix}1&&&\\b_2/A_1&1&&\\&b_3/A_2&1&\\&&\ddots&\\&&b_n/A_{n-1}&1\end{pmatrix},
\quad U = \begin{pmatrix}A_1&d_1&&\\&A_2&d_2&\\&&\ddots&\\&&&A_n\end{pmatrix},$$

так что $\det A = c_1\prod_{i=2}^{n}A_i$ — определитель считается *точно
по ходу* прогонки.

**Корректность и устойчивость прогонки.** Прогонка называется
корректной, если знаменатели $A_i \neq 0$ при всех $i$, и устойчивой,
если $|\delta_i| \le 1$ (коэффициенты не растут при проходе).

**Теорема 2.2.** Пусть в коэффициенты $b_i \neq 0$ при
$i = 2,\dots,n-1$, и пусть выполнено условие диагонального преобладания

$$|c_i| \ge |b_i| + |d_i|, \qquad i = 1,\dots,n. \tag{2.24}$$

Тогда прогонка корректна и устойчива
($A_i \neq 0$, $|\delta_i| < 1$).

Доказательство — методом индукции одновременного установления обоих
фактов: база $i = 1$ ($A_1 = c_1 \neq 0$, $|\delta_1| \le 1$ из (2.24)),
шаг $i$ использует $|c_i + b_i\delta_{i-1}| \ge |c_i| - |b_i||\delta_{i-1}|
\ge |c_i| - |b_i| \ge |d_i| \ge 0$ .